# Relational GCN for Link Prediction on FB15k-237

Knowledge Graph Completion on FB15k-237: Encoder-decoder knowledge graph completion using RGCN and DistMult score function. This notebook implements the approach with `RGCNConv / DistMult` inside a `K3RGCNLinkPred` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `RGCNConv / DistMult` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers

title = "Relational GCN for Link Prediction"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. RGCN Link Predictor
class K3RGCNLinkPred(keras.Model):
    def __init__(self, num_nodes, hidden_channels, num_relations):
        super().__init__()
        self.num_nodes = num_nodes
        self.node_emb = layers.Embedding(num_nodes, hidden_channels)
        self.conv = k3_layers.RGCNConv(hidden_channels, hidden_channels, num_relations=num_relations)

    def call(self, edge_index, edge_type):
        x = self.node_emb(ops.arange(self.num_nodes))
        return self.conv(x, edge_index, edge_type)

k3_model = K3RGCNLinkPred(num_nodes=100, hidden_channels=32, num_relations=10)

dummy_edges = ops.convert_to_tensor([[0, 1, 2], [1, 2, 0]], dtype="int64")
dummy_types = ops.convert_to_tensor([0, 1, 2], dtype="int64")

z = k3_model(dummy_edges, dummy_types)
print(f"Extracted relational node embeddings: {z.shape}")

print("\n✓ K3-Node RGCN Link Prediction execution completed successfully!")